# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** Here one row is one line item within the order. So order one that has two line items would have two rows in the result.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order','vendor_id']) #record path says we want the line items to supply the rows
#meta says which parent field we want copied with each row which is told to us that we want order and vendor_id
# TODO: assert the row count and the quantity total
print(len(lines))
assert len(lines) == 4 #makes sure the row count equals 4
print(lines['qty'].sum())
assert lines['qty'].sum() == 7 #sums quantity and ensures it totals to 7

4
7


**Markdown:** Here we flattened the orders so that instead of getting rows by orders we will get rows by line item. This is better because it give us more detail into quantities and items bought instead of just an overarching order.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
joined = lines.merge(pd.DataFrame(vendors_json), on = 'vendor_id', how = 'left', indicator = True) #join on the vendor id which is the overlap between the two sets of data, left join so that we can see any non-matching values
unmatched = joined[joined['_merge'] == 'left_only'] #finding which vendor didn't match
print(unmatched[['vendor_id','qty']]) #printing the vendor that didn't match and the units attached to them, also called their qty

  vendor_id  qty
3      V-07    3


**Markdown:** We joined the two data sets which in turn revealed that there was an unmatched vendor id between the two data sets. We did this by using left join to make sure that we are properly combining the two data sets and noting where information may be missing.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
vendor_totals = joined.groupby('name',dropna=False)['qty'].sum() #groups by vendor name and the sum of quanities for each
print(vendor_totals)
#The unmatched vendor appears as NaN in the table because it did not have a name associated with its vendor id, and therefore it is an unknown value to python. I decided to keep this in the table because the orders are still revelant to the analysis even if we do not know the vendor name. It is important to keep this unmatched value in so that we don't skew the data.

name
Cav Merch       1
Hoos Burgers    3
NaN             3
Name: qty, dtype: int64


**Markdown:** Found the total quanity each vendor sold and reported it with their vendor name. However we discovered in the question before that one vendor doesn't have a recognized vendor name, so this will be reported as NaN. It is important to still report this value because even though we are misisng some data, removing it completly would result in an inaccurate picture of the data set. Instead we acknowledge the missing information by denoting it Nan but keep it in our calculations as to keep everything even and correct.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
zone_totals = joined.groupby('zone')['qty'].sum().sort_values(ascending=False) #grouping by zone totals as seen by selection of zone and qty sum, sorting descending so that index 0 will be the zone that sold the most
print(zone_totals)
print(f'The zone that sold the most items was', zone_totals.index[0],'with', zone_totals.iloc[0], 'units.') #references zone totals to print which zone sold the most items

zone
A    4
Name: qty, dtype: int64
The zone that sold the most items was A with 4 units.


**Explanation:** The zone total was only 4 which indicates that the unmatched vendor has no zone because we know from previous question that there should be a total of 7 values. So the missing 3 units are with the unmatched vendor which has no zone and therefore is not included.

**Markdown:** Here we grouped the data into zones to see which sold the most items. However there could be some confusion since the zone totals don't equal our overall item total. We must remeber that not all values are put into zones so this could make our information look like it is not right, however this is why it is important to implement assertions and checks along the way so that we could see there are 7 quantities overall and since this value is only 4, we know there are some not in a zone.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
for order in ragged_json:
  if 'lines' not in order:
    order['lines']=[] #adding an empty lines list for orders that don't have a line value so that when we normalize the program doesn't crash
#since we are making empty lines, any order that doesn't have items won't show up as a row, therefore order 5 will be dropped from the printed table

ragged_lines = pd.json_normalize(ragged_json, record_path = 'lines', meta=['order','vendor_id']) #says that for ragged_json, lines will supply the rows, and the parent fields we want copied by each row are vendor id and order
ragged_lines['qty_given'] = ragged_lines['qty'].notna() #this creates a new column which helps us clarify that the quantity was truly not 0, but instead just a missing value
print(ragged_lines)

    item  qty order vendor_id  qty_given
0   Soda  2.0     4      V-01       True
1  Fries  NaN     6      V-01      False


**Markdown:** Flattened the data so that we could produce a data frame and show when a value is clearly missing or zero. First did this by filling in gaps of the data with empty values instead of no values so that the program didn't crash. Then we made an extra column that clarified whether a quantity was given to illustrate is the quantity was missing or zero. If you get NaN and quantity given is false then you know the value is truly missing as opposed to a zero.

### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**Answer a:** One row per line item is useful because some orders have multiple items that were bought. This makes it easy to see total units sold by each vendor. If we had instead done row by order, the vendors sales would be skewed because then we can't see size of each order, just that an order was made.

**Answer b:** A quantity of zero means that no units were sold while a missing quantity means that an unknown number of units were sold. This matters because it impacts totals and sales. If you treat a missing value the same as a zero this would be a practical issue because you underestimate sales even though you would have less inventory now. This would cause issues down that line by producing incorrect totals, averages, or inventory calculations.